# 📥 Notebook 1 — Data Collection & Structural Cleaning
**Project:** Citizen Grievance Analysis  
**Dataset:** NYC 311 Service Requests  


### What this notebook does:
- Loads the raw CSV from `data/raw/`
- Inspects shape, columns, null values
- Selects relevant columns
- Fixes missing values
- Parses dates & engineers new features
- Saves cleaned data → `data/cleaned/grievance_cleaned.csv`

---

##  Setup Paths

In [11]:
# Paths for data collection and storage ─────────────────────
RAW_DATA_PATH = '../data/raw/311_service_requests_sample.csv'
OUTPUT_DIR    = '../data/cleaned/'

## Load the Raw CSV

In [ ]:
import pandas as pd
df = pd.read_csv(RAW_DATA_PATH, low_memory=False)

print(f'Rows    : {df.shape[0]:,}')
print(f'Columns : {df.shape[1]}')
df.head(3)

Rows    : 10,000
Columns : 53


,Unique Key,Created Date,Closed Date,Agency,Agency Name,Complaint Type,Descriptor,Location Type,Incident Zip,Incident Address,...,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Garage Lot Name,Ferry Direction,Ferry Terminal Name,Latitude,Longitude,Location
0,32310363,12/31/2015 11:59:45 PM,01/01/2016 12:55:15 AM,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,10034.0,71 VERMILYEA AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.865682,-73.923501,"(40.86568153633767, -73.92350095571744)"
1,32309934,12/31/2015 11:59:44 PM,01/01/2016 01:26:57 AM,NYPD,New York City Police Department,Blocked Driveway,No Access,Street/Sidewalk,11105.0,27-07 23 AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.775945,-73.915094,"(40.775945312321085, -73.91509393898605)"
2,32309159,12/31/2015 11:59:29 PM,01/01/2016 04:51:03 AM,NYPD,New York City Police Department,Blocked Driveway,No Access,Street/Sidewalk,10458.0,2897 VALENTINE AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.870325,-73.888525,"(40.870324522111424, -73.88852464418646)"


##  Check Missing Values

In [4]:
null_df = pd.DataFrame({
    'Null Count': df.isnull().sum(),
    'Null %'    : (df.isnull().sum() / len(df) * 100).round(2)
})

# Show only columns that HAVE nulls
print(null_df[null_df['Null Count'] > 0].to_string())

                                Null Count  Null %
Closed Date                             57    0.57
Descriptor                             178    1.78
Incident Zip                            66    0.66
Incident Address                      1107   11.07
Street Name                           1107   11.07
Cross Street 1                        1248   12.48
Cross Street 2                        1264   12.64
Intersection Street 1                 8899   88.99
Intersection Street 2                 8913   89.13
Address Type                            72    0.72
City                                    66    0.66
Landmark                              9994   99.94
Facility Type                           55    0.55
Resolution Action Updated Date          55    0.55
X Coordinate (State Plane)              86    0.86
Y Coordinate (State Plane)              86    0.86
School or Citywide Complaint         10000  100.00
Vehicle Type                         10000  100.00
Taxi Company Borough           

##  Select Only Useful Columns

In [5]:
# We only need these 10 columns for NLP work
NLP_COLS = [
    'Unique Key',
    'Created Date',
    'Closed Date',
    'Agency Name',
    'Complaint Type',        # ← This is our TARGET LABEL for classification
    'Descriptor',            # ← Main complaint text
    'Location Type',
    'Borough',
    'Status',
    'Resolution Description' # ← How it was resolved
]

df_nlp = df[NLP_COLS].copy()
print(f'Reduced to {df_nlp.shape[1]} columns, {df_nlp.shape[0]:,} rows')
df_nlp.head(3)

Reduced to 10 columns, 10,000 rows


,Unique Key,Created Date,Closed Date,Agency Name,Complaint Type,Descriptor,Location Type,Borough,Status,Resolution Description
0,32310363,12/31/2015 11:59:45 PM,01/01/2016 12:55:15 AM,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,MANHATTAN,Closed,The Police Department responded and upon arriv...
1,32309934,12/31/2015 11:59:44 PM,01/01/2016 01:26:57 AM,New York City Police Department,Blocked Driveway,No Access,Street/Sidewalk,QUEENS,Closed,The Police Department responded to the complai...
2,32309159,12/31/2015 11:59:29 PM,01/01/2016 04:51:03 AM,New York City Police Department,Blocked Driveway,No Access,Street/Sidewalk,BRONX,Closed,The Police Department responded and upon arriv...


## Fix Missing Values

In [6]:
TEXT_COLS = [
    'Descriptor', 'Resolution Description',
    'Complaint Type', 'Agency Name', 'Location Type'
]

for col in TEXT_COLS:
    df_nlp[col] = df_nlp[col].fillna('unknown')

# Drop rows where Complaint Type is missing
# (it is our label — we cannot train without it)
df_nlp = df_nlp[df_nlp['Complaint Type'] != 'unknown']

print(f'Rows after fixing nulls: {len(df_nlp):,}')
print('Remaining nulls:')
print(df_nlp.isnull().sum())

Rows after fixing nulls: 10,000
Remaining nulls:
Unique Key                 0
Created Date               0
Closed Date               57
Agency Name                0
Complaint Type             0
Descriptor                 0
Location Type              0
Borough                    0
Status                     0
Resolution Description     0
dtype: int64


## Parse Dates & Engineer Features

In [7]:
df_nlp['Created Date'] = pd.to_datetime(df_nlp['Created Date'], errors='coerce')
df_nlp['Closed Date']  = pd.to_datetime(df_nlp['Closed Date'],  errors='coerce')

# How many hours to resolve the complaint?
df_nlp['resolution_hours'] = (
    (df_nlp['Closed Date'] - df_nlp['Created Date'])
    .dt.total_seconds() / 3600
)

# What hour of the day was complaint filed?
df_nlp['hour_created'] = df_nlp['Created Date'].dt.hour

# What day of the week?
df_nlp['day_of_week'] = df_nlp['Created Date'].dt.day_name()

print('New columns added: resolution_hours, hour_created, day_of_week')
df_nlp[['Created Date','Closed Date','resolution_hours','hour_created','day_of_week']].head()

New columns added: resolution_hours, hour_created, day_of_week


,Created Date,Closed Date,resolution_hours,hour_created,day_of_week
0,2015-12-31 23:59:45,2016-01-01 00:55:15,0.925000,23,Thursday
1,2015-12-31 23:59:44,2016-01-01 01:26:57,1.453611,23,Thursday
2,2015-12-31 23:59:29,2016-01-01 04:51:03,4.859444,23,Thursday
3,2015-12-31 23:57:46,2016-01-01 07:43:13,7.757500,23,Thursday
4,2015-12-31 23:56:58,2016-01-01 03:24:42,3.462222,23,Thursday


##  Create Combined Text Column

In [8]:
# Merge complaint type + descriptor + resolution into one text field
# This gives NLP models more signal to work with
df_nlp['combined_text'] = (
    df_nlp['Complaint Type'].astype(str)         + ' ' +
    df_nlp['Descriptor'].astype(str)             + ' ' +
    df_nlp['Resolution Description'].astype(str)
)

print('Sample combined_text:')
print(df_nlp['combined_text'].iloc[0])

Sample combined_text:
Noise - Street/Sidewalk Loud Music/Party The Police Department responded and upon arrival those responsible for the condition were gone.


##  Quick Summary & Save

In [9]:
print('=== FINAL CLEANED DATASET SUMMARY ===')
print(f'Total rows    : {len(df_nlp):,}')
print(f'Total columns : {df_nlp.shape[1]}')
print(f'Unique complaint types (labels): {df_nlp["Complaint Type"].nunique()}')
print(f'Boroughs      : {df_nlp["Borough"].unique().tolist()}')
print(f'Date range    : {df_nlp["Created Date"].min()} → {df_nlp["Created Date"].max()}')

# Save output for Notebook 2
SAVE_PATH = OUTPUT_DIR + 'grievance_cleaned.csv'
df_nlp.to_csv(SAVE_PATH, index=False)
print(f'\n✅ Saved → {SAVE_PATH}')
print('👉 Now open Notebook 2: 02_text_preprocessing.ipynb')

=== FINAL CLEANED DATASET SUMMARY ===
Total rows    : 10,000
Total columns : 14
Unique complaint types (labels): 19
Boroughs      : ['MANHATTAN', 'QUEENS', 'BRONX', 'BROOKLYN', 'Unspecified', 'STATEN ISLAND']
Date range    : 2015-12-20 17:29:35 → 2015-12-31 23:59:45

✅ Saved → ../data/cleaned/grievance_cleaned.csv
👉 Now open Notebook 2: 02_text_preprocessing.ipynb
